# Project 1: Market Basket Analysis with NLP & Association Mining
## Academic Case Study & Portfolio Walkthrough

### Objectives:
1. **Market Basket Analysis**: Mine transaction data using **Apriori** and **FP-Growth** algorithms to uncover frequent co-purchasing patterns.
2. **NLP Intelligence**: Apply Natural Language Processing (TF-IDF, Cosine Similarity, VADER Sentiment, Aspect Extraction) to understand product descriptions and customer reviews.
3. **Hybrid Recommendation Engine**: Combine transactional Lift with semantic similarity to deliver AI-explainable product recommendations and promotional bundles.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
sys.path.insert(0, os.path.abspath('..'))
from src.nlp_engine import NLPEngine
from src.market_basket_miner import MarketBasketMiner
from src.hybrid_recommender import HybridRecommender
from src.evaluate_recommender import ModelEvaluator

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print("Libraries loaded successfully.")

## 1. Exploratory Data Analysis: High-Volume Supermarket Datasets

In [ ]:
df_products = pd.read_csv('../data/products.csv')
df_train = pd.read_csv('../data/train_transactions.csv')
df_test = pd.read_csv('../data/test_transactions.csv')
df_reviews = pd.read_csv('../data/reviews.csv')

print(f"Total Products: {len(df_products):,}")
print(f"Train Transactions: {len(df_train):,}")
print(f"Test Transactions: {len(df_test):,}")
print(f"Customer Reviews: {len(df_reviews):,}")

# Department distribution
plt.figure(figsize=(10, 4))
df_products['category'].value_counts().plot(kind='bar', color='teal')
plt.title('Product Distribution across Supermarket Departments')
plt.xlabel('Department')
plt.ylabel('Product Count')
plt.xticks(rotation=45, ha='right')
plt.show()

## 2. Natural Language Processing (NLP) Engine
Extracting semantic similarity from product descriptions and sentiment polarity from customer reviews.

In [ ]:
nlp = NLPEngine(data_dir='../data')
nlp.fit_all()

# Inspect NLP similarity for Penne Pasta
pasta_id = "PROD_001"
sim_items = nlp.get_top_nlp_similar_products(pasta_id, top_n=5)
pd.DataFrame(sim_items)

## 3. Association Rule Mining: Apriori vs. FP-Growth
Mining frequent itemsets and profiling algorithmic execution speed.

In [ ]:
miner = MarketBasketMiner(data_dir='../data')
miner.fit_all(min_support=0.015, min_lift=1.2)

# Display Top Association Rules
cols = ['antecedent_names', 'consequent_names', 'support', 'confidence', 'lift']
miner.rules_df[cols].head(10)

## 4. Hybrid Recommendations & Promotional Bundle Generation
Fusing statistical Lift with semantic relevance and customer voice.

In [ ]:
recommender = HybridRecommender(data_dir='../data')
recommender.fit(min_support=0.015, min_lift=1.2)

# Generate recommendations for Pasta
recs = recommender.recommend_for_product("PROD_001", top_n=4)
for r in recs:
    print(f"Product: {r['product_name']} | Lift: {r['transaction_lift']}x | Hybrid Score: {r['hybrid_score']}")
    print(f"Explanation: {r['explanation']}\n")

# Generate promotional bundle
bundle = recommender.generate_promotional_bundle("PROD_001", bundle_size=4, discount_percent=15)
print(f"Bundle Name: {bundle['bundle_name']}")
print(f"Regular Price: ${bundle['regular_total_price']} -> Bundle Price: ${bundle['bundle_promotional_price']} (Save ${bundle['customer_savings']})")

## 5. Rigorous Out-of-Sample Holdout Testing
Testing models against 1,000 unseen shopping baskets.

In [ ]:
eval_metrics_path = '../reports/model_evaluation_metrics.csv'
if os.path.exists(eval_metrics_path):
    df_eval = pd.read_csv(eval_metrics_path)
    display(df_eval)
else:
    evaluator = ModelEvaluator(data_dir='../data', reports_dir='../reports')
    evaluator.setup()
    df_eval = evaluator.evaluate_all_models(test_sample_size=500, k_values=[3, 5])
    display(df_eval)